# Hold-out и сравнение трёх моделей
Вход: data/processed/train_features.parquet (сохранён в 04_installments)

**Правило hold-out:** 20% клиентов откладываем и не трогаем до финальной оценки.
Всё остальное (CV, подбор гиперпараметров, выбор порога, калибровка) делается на dev.
Если подглядывать в hold-out при выборе модели, он перестаёт быть честной оценкой.

In [ ]:
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import precision_recall_curve, roc_curve

sys.path.insert(0, "..")
from src.data import load_features, split_dev_holdout
from src.features import ID_COL, TARGET_COL
from src.metrics import report
from src.models import (cv_predict, make_lgbm, make_logreg,
                        make_random_forest, to_lgbm)

PROCESSED = Path("../data/processed")

## 1. Разбиение dev / hold-out

In [ ]:
df = load_features(PROCESSED / "train_features.parquet")
dev, hold = split_dev_holdout(df, PROCESSED / "holdout_ids.csv", holdout_size=0.2)

print(f"dev:  {dev.shape[0]:>7} строк | доля дефолтов {dev[TARGET_COL].mean():.4f}")
print(f"hold: {hold.shape[0]:>7} строк | доля дефолтов {hold[TARGET_COL].mean():.4f}")
print("ID не пересекаются:", set(dev[ID_COL]).isdisjoint(hold[ID_COL]))

# дальше hold нигде не используется до финального ноутбука
y = dev[TARGET_COL]
X = dev.drop(columns=[TARGET_COL, ID_COL])
print("Признаков:", X.shape[1])

## 2. Три модели на одних и тех же фолдах
Время: логрег и LightGBM по несколько минут, случайный лес дольше всех.
Если слишком долго, для отладки поставь n_estimators=100 у леса.

In [ ]:
results = {}


def run(name, make_model, X_):
    t = time.time()
    oof, fold_auc = cv_predict(make_model, X_, y)
    results[name] = (oof, fold_auc, time.time() - t)
    print(f"{name}: AUC по фолдам {fold_auc.mean():.4f} ± {fold_auc.std():.4f} "
          f"({time.time() - t:.0f} c)")


run("LogReg", lambda: make_logreg(X), X)

In [ ]:
run("LightGBM", make_lgbm, to_lgbm(X))

In [ ]:
run("RandomForest", lambda: make_random_forest(X), X)

## 3. Сводная таблица
Метрики по OOF-предсказаниям (каждая строка предсказана моделью, которая её не видела).
Они чуть отличаются от среднего по фолдам: это нормально.

In [ ]:
rows = []
for name, (oof, fold_auc, sec) in results.items():
    r = report(name, y, oof)
    r["AUC folds mean"] = round(fold_auc.mean(), 4)
    r["AUC folds std"] = round(fold_auc.std(), 4)
    r["fit, c"] = round(sec)
    rows.append(r)

summary = pd.DataFrame(rows).set_index("model")
summary

In [ ]:
# Попарные разницы по фолдам: стабильно ли одна модель лучше другой
pairs = [("LightGBM", "LogReg"), ("LightGBM", "RandomForest"), ("RandomForest", "LogReg")]
diff = pd.DataFrame({
    f"{a} - {b}": {
        "mean": (results[a][1] - results[b][1]).mean(),
        "min": (results[a][1] - results[b][1]).min(),
        "max": (results[a][1] - results[b][1]).max(),
    }
    for a, b in pairs
}).T.round(4)
diff

## 4. Кривые

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for name, (oof, _, _) in results.items():
    fpr, tpr, _ = roc_curve(y, oof)
    axes[0].plot(fpr, tpr, label=name)
    prec, rec, _ = precision_recall_curve(y, oof)
    axes[1].plot(rec, prec, label=name)

axes[0].plot([0, 1], [0, 1], "k--", alpha=0.4)
axes[0].set(title="ROC (OOF на dev)", xlabel="FPR", ylabel="TPR")
axes[1].axhline(y.mean(), color="k", ls="--", alpha=0.4, label="случайная модель")
axes[1].set(title="Precision-Recall (OOF на dev)", xlabel="Recall", ylabel="Precision")
for ax in axes:
    ax.legend()
plt.tight_layout()
plt.show()

## 5. Сохраняем OOF-предсказания
Пригодятся для калибровки, выбора порога и анализа ошибок без повторного обучения.

In [ ]:
oof_df = pd.DataFrame({ID_COL: dev[ID_COL], TARGET_COL: y})
for name, (oof, _, _) in results.items():
    oof_df[name] = oof
oof_df.to_parquet(PROCESSED / "oof_dev.parquet", index=False)
summary.to_csv(PROCESSED / "model_comparison.csv")
print("Сохранено")